In [1]:
import pandas as pd

df = pd.DataFrame({
    "Player": ["Baba Rahman", "Derrick Kohn"],

    # Availability
    "Matches": [25, 29],
    "Minutes": [1920, 1899],
    "90s": [21.3, 21.1],

    # Attacking
    "Goals": [2, 1],
    "Assists": [2, 2],
    "xG": [1.43, 2.15],
    "xA": [1.87, 3.78],
    "Shots": [14, 28],
    "Chances_Created": [13, 25],
    "Big_Chances_Created": [4, 5],

    # Passing
    "Pass_Accuracy": [78.8, 76.3],
    "Successful_Passes": [733, 425],
    "Accurate_Long_Balls": [33, 32],
    "Long_Ball_Accuracy": [35.5, 36.8],

    # Crossing
    "Successful_Crosses": [9, 31],
    "Cross_Accuracy": [19.1, 27.0],

    # Possession
    "Successful_Dribbles": [5, 11],
    "Dribble_Success": [31.2, 42.3],
    "Duels_Won": [78, 67],
    "Duel_Win_Pct": [51.3, 44.1],
    "Aerial_Duels_Won": [23, 6],
    "Aerial_Win_Pct": [53.5, 50.0],
    "Touches": [1508, 1138],
    "Touches_Opp_Box": [38, 33],
    "Dispossessed": [8, 23],
    "Fouls_Won": [21, 16],

    # Defensive
    "Defensive_Contributions": [114, 87],
    "Tackles": [29, 34],
    "Interceptions": [19, 8],
    "Blocked_Shots": [5, 5],
    "Recoveries": [69, 88],
    "Poss_Won_Final_3rd": [3, 5],
    "Dribbled_Past": [11, 9],
    "Clearances": [61, 40],
    "Clean_Sheets": [9, 3],
    "Goals_Conceded_On_Pitch": [18, 32],
    "xGA_On_Pitch": [22.0, 29.9],

    # Discipline
    "Yellow_Cards": [3, 5],
    "Red_Cards": [0, 1]
})

print(df)

         Player  Matches  Minutes   90s  Goals  Assists    xG    xA  Shots  \
0   Baba Rahman       25     1920  21.3      2        2  1.43  1.87     14   
1  Derrick Kohn       29     1899  21.1      1        2  2.15  3.78     28   

   Chances_Created  ...  Blocked_Shots  Recoveries  Poss_Won_Final_3rd  \
0               13  ...              5          69                   3   
1               25  ...              5          88                   5   

   Dribbled_Past  Clearances  Clean_Sheets  Goals_Conceded_On_Pitch  \
0             11          61             9                       18   
1              9          40             3                       32   

   xGA_On_Pitch  Yellow_Cards  Red_Cards  
0          22.0             3          0  
1          29.9             5          1  

[2 rows x 40 columns]


In [2]:
df.head()

,Player,Matches,Minutes,90s,Goals,Assists,xG,xA,Shots,Chances_Created,...,Blocked_Shots,Recoveries,Poss_Won_Final_3rd,Dribbled_Past,Clearances,Clean_Sheets,Goals_Conceded_On_Pitch,xGA_On_Pitch,Yellow_Cards,Red_Cards
0,Baba Rahman,25,1920,21.3,2,2,1.43,1.87,14,13,...,5,69,3,11,61,9,18,22.0,3,0
1,Derrick Kohn,29,1899,21.1,1,2,2.15,3.78,28,25,...,5,88,5,9,40,3,32,29.9,5,1


In [8]:
df.to_csv("comparison_data.csv", index=False)

from google.colab import files
files.download("comparison_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Create Per-90 Metrics

In [3]:
per90_cols = [
    "Goals",
    "Assists",
    "Shots",
    "Chances_Created",
    "Successful_Crosses",
    "Successful_Dribbles",
    "Duels_Won",
    "Tackles",
    "Interceptions",
    "Recoveries",
    "Clearances"
]

for col in per90_cols:
    df[f"{col}_Per90"] = df[col] / df["90s"]

# Create Category Scores

In [4]:
df["Attack_Score"] = (
    df["Goals_Per90"] * 4 +
    df["Assists_Per90"] * 3 +
    df["xA"] * 2 +
    df["Chances_Created_Per90"]
)

df["Defense_Score"] = (
    df["Tackles_Per90"] +
    df["Interceptions_Per90"] +
    df["Recoveries_Per90"] +
    df["Clearances_Per90"]
)

df["Possession_Score"] = (
    df["Pass_Accuracy"] +
    df["Duel_Win_Pct"] +
    df["Aerial_Win_Pct"]
)

df["Discipline_Score"] = (
    100 -
    (df["Yellow_Cards"] * 5) -
    (df["Red_Cards"] * 20)
)

# Create Final Readiness Index

In [7]:
from sklearn.preprocessing import MinMaxScaler

score_cols = [
    "Attack_Score",
    "Defense_Score",
    "Possession_Score",
    "Discipline_Score"
]

scaler = MinMaxScaler()

df[score_cols] = scaler.fit_transform(df[score_cols])

df["Readiness_Index"] = (
    0.30 * df["Attack_Score"] +
    0.30 * df["Defense_Score"] +
    0.25 * df["Possession_Score"] +
    0.15 * df["Discipline_Score"]
) * 100

print(
    df[["Player", "Readiness_Index"]]
    .sort_values("Readiness_Index", ascending=False)
)

         Player  Readiness_Index
0   Baba Rahman             70.0
1  Derrick Kohn             30.0
